# Step 9 (Phase 4) — Escalation Predictor

Trains an XGBoost model to answer: "given an incident's first few readings,
will it turn out severe?" Uses the 5-clue table built by
`src/ml/escalation_prep.py` (Phase 4, Step 2).

Trains on 824 early-window readings from 85 incidents.
Tests on 259 sealed readings from 22 different, later incidents —
never seen during training.

## Cell 1 — Load the 4 files

Loads the practice table (`X_train`/`y_train`) and the sealed exam table
(`X_test`/`y_test`) saved earlier, plus the label sheet that tells us what
each of the 5 columns actually means.

In [19]:
import json
from pathlib import Path

import numpy as np
import xgboost as xgb
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix

import os
os.chdir(Path("..").resolve() if Path.cwd().name == "notebooks" else Path.cwd())

ML_DIR = Path("ml_models")

X_train = np.load(ML_DIR / "X_escalation_train.npy")
X_test  = np.load(ML_DIR / "X_escalation_test.npy")
y_train = np.load(ML_DIR / "y_escalation_train.npy")
y_test  = np.load(ML_DIR / "y_escalation_test.npy")

with open(ML_DIR / "escalation_feature_cols.json") as f:
    feature_cols = json.load(f)

print("Train:", X_train.shape, "  Test:", X_test.shape)
print("Feature order:", feature_cols)

Train: (1354, 5)   Test: (424, 5)
Feature order: ['votes', 'ensemble_score', 'zscore_value', 'iforest_score', 'lstm_error']


## Cell 2 — Sanity check before training

Just double-checks the loaded files match what `escalation_prep.py` printed
earlier (824 train rows / 200 severe, 259 test rows / 96 severe). A quick
habit to catch a stale or wrong file before wasting time training on it.

In [20]:
print(f"Train: {y_train.sum()} severe / {len(y_train)} rows ({100*y_train.mean():.1f}%)")
print(f"Test:  {y_test.sum()} severe / {len(y_test)} rows ({100*y_test.mean():.1f}%)")

Train: 324 severe / 1354 rows (23.9%)
Test:  156 severe / 424 rows (36.8%)


## Cell 3 — Train the model

`scale_pos_weight` tells the model "treat getting a severe case wrong as
roughly N times more costly than getting a non-severe case wrong" — this
stops it from taking the lazy shortcut of always guessing "not severe"
(which would already be right ~76% of the time, but useless).

`max_depth=3` keeps each tree shallow on purpose. With only 824 rows and
5 clues, a deep tree would just memorize the practice set instead of
learning a real pattern — like a student who memorizes the exact practice
questions instead of understanding the topic, then fails on new questions.

In [21]:
n_pos = y_train.sum()
n_neg = len(y_train) - n_pos
scale_pos_weight = n_neg / n_pos
print(f"scale_pos_weight = {scale_pos_weight:.2f}")

model = xgb.XGBClassifier(
    n_estimators=100,       # how many trees to build
    max_depth=3,            # keep trees shallow — avoid memorizing 824 rows
    learning_rate=0.1,      # how big a correction each tree is allowed to make
    scale_pos_weight=scale_pos_weight,
    eval_metric="logloss",
    random_state=42,
)

model.fit(X_train, y_train)
print("Trained.")

scale_pos_weight = 3.18
Trained.


## Cell 4 — Grade it honestly against the sealed test rows

This is the only place the model's guesses ever get compared to the 259
test answers. Precision, recall, and F1 mean the same thing here as they
did in Phase 3's `05_evaluation.ipynb` — just applied to "will this incident
turn severe" instead of "is this reading anomalous."

In [22]:
y_pred = model.predict(X_test)

print(f"Precision: {precision_score(y_test, y_pred):.3f}")
print(f"Recall:    {recall_score(y_test, y_pred):.3f}")
print(f"F1:        {f1_score(y_test, y_pred):.3f}")
print("\nConfusion matrix (rows=actual, cols=predicted):")
print(confusion_matrix(y_test, y_pred))

Precision: 0.413
Recall:    0.500
F1:        0.452

Confusion matrix (rows=actual, cols=predicted):
[[157 111]
 [ 78  78]]


## Cell 5 — Save the trained model

Same pattern as your Z-Score/Isolation Forest/LSTM files in `ml_models/` —
a model file plus a small config file recording which 5 columns it expects
and in what order, so nothing gets mixed up when it's loaded again later.


In [23]:
model.save_model(ML_DIR / "xgb_escalation.json")

with open(ML_DIR / "xgb_escalation_config.json", "w") as f:
    json.dump({"feature_cols": feature_cols, "scale_pos_weight": scale_pos_weight}, f, indent=2)

print("Saved xgb_escalation.json + xgb_escalation_config.json to ml_models/")

Saved xgb_escalation.json + xgb_escalation_config.json to ml_models/


## Cell 6 — Which clues did the model actually lean on?

`model.feature_importances_` gives one number per clue, showing roughly how
often that clue was used to make a split across all the trees. Higher =
the model leaned on it more. If everything comes out roughly equal and low,
that's a sign the model didn't find much of a real pattern in any single
clue — more evidence it's closer to guessing than learning.

In [24]:
importances = model.feature_importances_

ranked = sorted(zip(feature_cols, importances), key=lambda x: x[1], reverse=True)

print("Feature importance (higher = leaned on more):")
for name, score in ranked:
    bar = "█" * int(score * 50)
    print(f"  {name:<16} {score:.3f}  {bar}")

Feature importance (higher = leaned on more):
  lstm_error       0.291  ██████████████
  iforest_score    0.259  ████████████
  votes            0.227  ███████████
  zscore_value     0.223  ███████████
  ensemble_score   0.000  


## Cell 7 — Build the SHAP explainer

`TreeExplainer` is built specifically for tree-based models like XGBoost —
it can calculate exact contributions efficiently instead of estimating
them. We compute SHAP values for every row in the test set at once.

In [25]:
import shap

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

print("SHAP values shape:", shap_values.shape)   # [n_test_rows, 5 features]
print("Base value (average prediction before any clues):", explainer.expected_value)

/Users/esanduepa/Desktop/Projects/anomaly-detection-platform/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


SHAP values shape: (424, 5)
Base value (average prediction before any clues): 9.009019e-05


## Cell 8 — Explain ONE real prediction, in plain terms

Picks a test row the model was confident about and prints, clue by clue,
how much it pushed the prediction up or down — the same numbers that
will later feed Alert.explanation_text in Phase 5.

In [26]:
import numpy as np

# Pick the test row the model was MOST confident was severe
probs = model.predict_proba(X_test)[:, 1]
i = int(np.argmax(probs))

print(f"Row {i} — model's predicted chance of severe: {probs[i]:.1%}")
print(f"Actual answer: {'severe' if y_test[i] == 1 else 'not severe'}\n")

print("Why the model guessed this way:")
base = explainer.expected_value
running = base
for name, value, contribution in sorted(
    zip(feature_cols, X_test[i], shap_values[i]),
    key=lambda x: abs(x[2]), reverse=True
):
    direction = "pushed UP →" if contribution > 0 else "pulled DOWN →"
    print(f"  {name:<16} = {value:<8.3f}  {direction} {contribution:+.3f}")
    running += contribution

print(f"\n  Starting point (average incident): {base:.3f}")
print(f"  Final score after all 5 clues:      {running:.3f}")

Row 49 — model's predicted chance of severe: 79.9%
Actual answer: severe

Why the model guessed this way:
  lstm_error       = 0.124     pushed UP → +0.672
  zscore_value     = 3.157     pushed UP → +0.444
  iforest_score    = -0.013    pushed UP → +0.267
  votes            = 2.000     pulled DOWN → -0.004
  ensemble_score   = 0.667     pulled DOWN → +0.000

  Starting point (average incident): 0.000
  Final score after all 5 clues:      1.380


## Cell 9 — Turn SHAP numbers into a plain-English sentence

This is the last piece of Step 4 — automatically building the kind of
explanation Alert.explanation_text and Alert.contributing_features
(src/models/alert.py) were built to hold, but empty until now.

Each of the 5 clues gets a short, human-readable description for "this
pushed toward severe" and "this pushed away from severe." We take the
top 2-3 clues that mattered most for THIS prediction and stitch them
into one sentence. ensemble_score is skipped here — it's always
redundant with votes (Cell 6 showed it gets 0 importance), so including
it would just be noise in the sentence.

In [27]:
FEATURE_DESCRIPTIONS = {
    "votes": {
        "up":   "an unusually high number of detectors agreed something was wrong",
        "down": "only a few detectors flagged anything unusual",
    },
    "zscore_value": {
        "up":   "the readings were far outside their normal statistical range",
        "down": "the readings stayed close to their normal statistical range",
    },
    "iforest_score": {
        "up":   "the overall combination of metrics looked highly unusual to the pattern-detection model",
        "down": "the overall combination of metrics looked fairly typical",
    },
    "lstm_error": {
        "up":   "the recent sequence of readings didn't match any normal pattern the system has learned",
        "down": "the recent sequence of readings still resembled normal patterns",
    },
}

def explain_prediction(idx, top_k=3):
    # Convert log-odds back to an actual percentage (see the 1.380 → 79.9% fix)
    raw_margin = explainer.expected_value + shap_values[idx].sum()
    probability = 1 / (1 + np.exp(-raw_margin))

    contributions = [
        c for c in zip(feature_cols, X_test[idx], shap_values[idx])
        if c[0] != "ensemble_score"          # redundant with votes — skip in the sentence
    ]
    contributions.sort(key=lambda x: abs(x[2]), reverse=True)
    top = contributions[:top_k]

    reasons = [
        FEATURE_DESCRIPTIONS[name]["up" if contribution > 0 else "down"]
        for name, value, contribution in top
    ]

    verdict = "likely to become severe" if probability >= 0.5 else "unlikely to become severe"
    sentence = f"This incident looks {verdict} ({probability:.0%} confidence), mainly because {reasons[0]}"
    if len(reasons) > 1:
        sentence += f", and {reasons[1]}"
    if len(reasons) > 2:
        sentence += f". A smaller factor: {reasons[2]}"
    sentence += "."

    return {
        "probability": round(float(probability), 3),
        "explanation_text": sentence,
        "contributing_features": [
            {"feature": name, "value": round(float(value), 4), "contribution": round(float(contribution), 4)}
            for name, value, contribution in top
        ],
    }

result = explain_prediction(i)   # reusing row i from Cell 8
print(result["explanation_text"])
print()
print(result["contributing_features"])

This incident looks likely to become severe (80% confidence), mainly because the recent sequence of readings didn't match any normal pattern the system has learned, and the readings were far outside their normal statistical range. A smaller factor: the overall combination of metrics looked highly unusual to the pattern-detection model.

[{'feature': 'lstm_error', 'value': 0.1238, 'contribution': 0.6721}, {'feature': 'zscore_value', 'value': 3.1573, 'contribution': 0.4443}, {'feature': 'iforest_score', 'value': -0.0135, 'contribution': 0.267}]


In [28]:
# Find a false positive: model said "severe", but it actually wasn't
wrong_indices = np.where((y_pred == 1) & (y_test == 0))[0]

if len(wrong_indices) > 0:
    wrong_i = int(wrong_indices[0])
    print(f"Row {wrong_i} — model said severe, but it actually wasn't\n")
    result = explain_prediction(wrong_i)
    print(result["explanation_text"])

Row 1 — model said severe, but it actually wasn't

This incident looks likely to become severe (51% confidence), mainly because the recent sequence of readings still resembled normal patterns, and the readings were far outside their normal statistical range. A smaller factor: the overall combination of metrics looked highly unusual to the pattern-detection model.
